In [82]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.weightstats import ttest_ind
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.stats.api as sms
import statsmodels.api as sm

## Following analysis uses the DPP method (not including MJF in low RSD)

In [83]:
dpw_beacon = pd.read_csv("raw beacon data/dpw_beacon_raw.csv")
mjf_beacon = pd.read_csv("raw beacon data/mjf_beacon_raw.csv")
pema_beacon = pd.read_csv("raw beacon data/pema_beacon_raw.csv")
pha_beacon = pd.read_csv("raw beacon data/pha_beacon_raw.csv")

In [84]:
no2 = pd.read_csv("low RSD ready data/no2_bysite.csv")
no2["datetime_utc"] = pd.to_datetime(no2["datetime_utc"])

In [85]:
low_rsd = pd.read_csv("low RSD data/no2_low_rsd_dpp.csv")
low_rsd["datetime_utc"] = pd.to_datetime(low_rsd["datetime_utc"])

### DPW

In [86]:
dpw_beacon.head()

,local_timestamp,epoch,datetime,node_file_id,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,node_id
0,2025-10-08 14:00:00,1.759957e+09,2025-10-08 21:00:00,6264147,0.078219,-0.019430,0.026894,-0.000242,NaN,NaN,276
1,2025-10-08 16:00:00,1.759964e+09,2025-10-08 23:00:00,6264814,0.080516,-0.016653,0.024906,0.001759,NaN,NaN,276
2,2025-10-08 17:00:00,1.759968e+09,2025-10-09 00:00:00,6265155,0.089586,-0.015961,0.024753,0.002580,NaN,NaN,276
3,2025-10-08 18:00:00,1.759972e+09,2025-10-09 01:00:00,6265693,0.112243,-0.013000,0.023088,0.005258,NaN,NaN,276
4,2025-10-08 20:00:00,1.759979e+09,2025-10-09 03:00:00,6266735,0.084116,-0.014280,0.022850,0.004924,NaN,NaN,276


In [87]:
dpw_beacon = dpw_beacon[["datetime", "co_wrk_aux", "no2_wrk_aux", "no_wrk_aux", "o3_wrk_aux", "rh", "temp"]].rename(columns={"datetime": "datetime_utc"})
dpw_beacon["datetime_utc"] = pd.to_datetime(dpw_beacon["datetime_utc"])
dpw_beacon.head()

,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp
0,2025-10-08 21:00:00,0.078219,-0.019430,0.026894,-0.000242,NaN,NaN
1,2025-10-08 23:00:00,0.080516,-0.016653,0.024906,0.001759,NaN,NaN
2,2025-10-09 00:00:00,0.089586,-0.015961,0.024753,0.002580,NaN,NaN
3,2025-10-09 01:00:00,0.112243,-0.013000,0.023088,0.005258,NaN,NaN
4,2025-10-09 03:00:00,0.084116,-0.014280,0.022850,0.004924,NaN,NaN


In [88]:
dpw_quant = no2[["datetime_utc", "dpw"]].rename(columns={"dpw": "dpw_quant_no2"})

dpw_combined = pd.merge(dpw_beacon, dpw_quant, on="datetime_utc", how="inner")

dpw_combined = dpw_combined.dropna(axis=0)

dpw_combined = (dpw_combined.sort_values("datetime_utc").reset_index(drop=True))

dpw_combined.to_csv("ML ready dpp no2 datasets/dpw_no2_test.csv", index=False)
dpw_combined.head()

,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,dpw_quant_no2
0,2025-10-24 16:00:00,0.069741,-0.023362,0.032977,-0.007498,27.649272,24.930835,6.9
1,2025-10-24 17:00:00,0.060418,-0.019520,0.027412,-0.002202,27.448088,24.270313,7.1
2,2025-10-24 18:00:00,0.062872,-0.018564,0.026829,-0.000036,27.671483,23.871093,9.2
3,2025-10-24 19:00:00,0.085247,-0.018020,0.028108,0.000576,27.203160,24.751195,10.7
4,2025-10-24 20:00:00,0.090493,-0.013422,0.022817,0.006713,30.394419,21.573989,11.9


In [89]:
n_obs = len(dpw_combined)
start_date = dpw_combined["datetime_utc"].min()
end_date = dpw_combined["datetime_utc"].max()

print(f"Number of observations: {n_obs}")
print(f"Start date: {start_date}")
print(f"End date: {end_date}")

Number of observations: 5254
Start date: 2025-10-24 16:00:00
End date: 2026-05-31 23:00:00


In [90]:
low_rsd_times = low_rsd["datetime_utc"]

dpw_low_rsd = dpw_combined[
    dpw_combined["datetime_utc"].isin(low_rsd_times)
].copy()

print(f"Rows retained: {len(dpw_low_rsd)}")

dpw_low_rsd.to_csv("ML ready dpp no2 datasets/dpw_no2_train.csv", index=False)
dpw_low_rsd.head()

Rows retained: 520


,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,dpw_quant_no2
274,2025-11-05 03:00:00,0.081277,-0.011200,0.022017,0.009709,36.745761,11.399394,25.9
276,2025-11-05 05:00:00,0.088662,-0.011188,0.020963,0.009330,39.001316,10.382738,24.4
320,2025-11-07 01:00:00,0.064958,-0.012370,0.021409,0.008287,35.232425,10.314069,21.4
329,2025-11-07 10:00:00,0.083511,-0.010479,0.023948,0.004475,48.717783,4.684071,21.0
369,2025-11-09 02:00:00,0.206416,-0.011584,0.023021,0.004421,40.128370,14.130043,26.8


### MJF

In [91]:
mjf_beacon.head()

,local_timestamp,epoch,datetime,node_file_id,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,node_id
0,2025-10-08 14:00:00,1.759957e+09,2025-10-08 21:00:00,6264040,0.038685,0.010176,0.032245,0.023106,44.322174,21.107882,250
1,2025-10-08 15:00:00,1.759961e+09,2025-10-08 22:00:00,6264332,0.044336,0.012089,0.030384,0.024390,45.924005,19.472705,250
2,2025-10-08 16:00:00,1.759964e+09,2025-10-08 23:00:00,6264812,0.037721,0.011648,0.030388,0.025641,42.519526,17.775414,250
3,2025-10-08 17:00:00,1.759968e+09,2025-10-09 00:00:00,6265250,0.044853,0.013164,0.029289,0.026693,43.078434,16.745692,250
4,2025-10-08 18:00:00,1.759972e+09,2025-10-09 01:00:00,6265517,0.058054,0.015633,0.028102,0.028091,47.216864,15.344422,250


In [92]:
mjf_beacon = mjf_beacon[["datetime", "co_wrk_aux", "no2_wrk_aux", "no_wrk_aux", "o3_wrk_aux", "rh", "temp"]].rename(columns={"datetime": "datetime_utc"})
mjf_beacon["datetime_utc"] = pd.to_datetime(mjf_beacon["datetime_utc"])
mjf_beacon.head()

,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp
0,2025-10-08 21:00:00,0.038685,0.010176,0.032245,0.023106,44.322174,21.107882
1,2025-10-08 22:00:00,0.044336,0.012089,0.030384,0.024390,45.924005,19.472705
2,2025-10-08 23:00:00,0.037721,0.011648,0.030388,0.025641,42.519526,17.775414
3,2025-10-09 00:00:00,0.044853,0.013164,0.029289,0.026693,43.078434,16.745692
4,2025-10-09 01:00:00,0.058054,0.015633,0.028102,0.028091,47.216864,15.344422


In [93]:
mjf_quant = no2[["datetime_utc", "mjf"]].rename(columns={"mjf": "mjf_quant_no2"})

mjf_combined = pd.merge(mjf_beacon, mjf_quant, on="datetime_utc", how="inner")

mjf_combined = mjf_combined.dropna(axis=0)

mjf_combined = (mjf_combined.sort_values("datetime_utc").reset_index(drop=True))

mjf_combined.to_csv("ML ready dpp no2 datasets/mjf_no2_test.csv", index=False)
mjf_combined.head()

,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,mjf_quant_no2
0,2025-10-09 22:00:00,0.064155,0.020512,0.023810,0.034294,26.422852,14.930490,7.9
1,2025-10-09 23:00:00,0.073046,0.022442,0.023621,0.033077,35.203873,11.598447,10.4
2,2025-10-10 00:00:00,0.109942,0.024321,0.024074,0.030811,45.765355,9.236830,17.3
3,2025-10-10 01:00:00,0.160503,0.024057,0.025299,0.028802,52.458024,8.069494,15.9
4,2025-10-10 02:00:00,0.138243,0.023416,0.028022,0.026411,55.868604,7.212855,13.4


In [94]:
n_obs = len(mjf_combined)
start_date = mjf_combined["datetime_utc"].min()
end_date = mjf_combined["datetime_utc"].max()

print(f"Number of observations: {n_obs}")
print(f"Start date: {start_date}")
print(f"End date: {end_date}")

Number of observations: 5557
Start date: 2025-10-09 22:00:00
End date: 2026-05-31 23:00:00


In [95]:
mjf_low_rsd = mjf_combined[
    mjf_combined["datetime_utc"].isin(low_rsd_times)
].copy()

print(f"Rows retained: {len(mjf_low_rsd)}")

mjf_low_rsd.to_csv("ML ready dpp no2 datasets/mjf_no2_train.csv", index=False)
mjf_low_rsd.head()

Rows retained: 499


,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,mjf_quant_no2
629,2025-11-05 03:00:00,0.076741,0.019974,0.026416,0.030299,37.952348,10.356573,10.0
631,2025-11-05 05:00:00,0.080633,0.021459,0.026271,0.029891,43.427655,8.896024,14.1
675,2025-11-07 01:00:00,0.053113,0.020109,0.026928,0.030314,41.645745,7.949142,9.8
684,2025-11-07 10:00:00,0.057022,0.020609,0.026846,0.023342,55.115621,2.068967,16.6
724,2025-11-09 02:00:00,0.090970,0.020145,0.026520,0.025684,49.036650,10.559099,9.5


### PEMA

In [96]:
pema_beacon.head()

,local_timestamp,epoch,datetime,node_file_id,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,node_id
0,2025-10-08 14:00:00,1.759957e+09,2025-10-08 21:00:00,6264099,0.059913,-0.006221,0.034353,0.016182,46.097850,21.931741,271
1,2025-10-08 15:00:00,1.759961e+09,2025-10-08 22:00:00,6264286,0.069298,-0.003658,0.032222,0.018806,49.665886,19.892090,271
2,2025-10-08 16:00:00,1.759964e+09,2025-10-08 23:00:00,6264719,0.072786,-0.003253,0.031325,0.020515,50.726533,17.193113,271
3,2025-10-08 17:00:00,1.759968e+09,2025-10-09 00:00:00,6265020,0.073071,-0.002592,0.031541,0.020788,48.247390,15.955815,271
4,2025-10-08 18:00:00,1.759972e+09,2025-10-09 01:00:00,6265472,0.073402,-0.000804,0.030422,0.022756,53.547474,14.404917,271


In [97]:
pema_beacon = pema_beacon[["datetime", "co_wrk_aux", "no2_wrk_aux", "no_wrk_aux", "o3_wrk_aux", "rh", "temp"]].rename(columns={"datetime": "datetime_utc"})
pema_beacon["datetime_utc"] = pd.to_datetime(pema_beacon["datetime_utc"])
pema_beacon.head()

,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp
0,2025-10-08 21:00:00,0.059913,-0.006221,0.034353,0.016182,46.097850,21.931741
1,2025-10-08 22:00:00,0.069298,-0.003658,0.032222,0.018806,49.665886,19.892090
2,2025-10-08 23:00:00,0.072786,-0.003253,0.031325,0.020515,50.726533,17.193113
3,2025-10-09 00:00:00,0.073071,-0.002592,0.031541,0.020788,48.247390,15.955815
4,2025-10-09 01:00:00,0.073402,-0.000804,0.030422,0.022756,53.547474,14.404917


In [98]:
pema_quant = no2[["datetime_utc", "pema"]].rename(columns={"pema": "pema_quant_no2"})

pema_combined = pd.merge(pema_beacon, pema_quant, on="datetime_utc", how="inner")

pema_combined = pema_combined.dropna(axis=0)

pema_combined = (pema_combined.sort_values("datetime_utc").reset_index(drop=True))

pema_combined.to_csv("ML ready dpp no2 datasets/pema_no2_test.csv", index=False)
pema_combined.head()

,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,pema_quant_no2
0,2025-10-09 23:00:00,0.205597,0.010674,0.028768,0.030975,45.077107,11.318088,6.7
1,2025-10-10 00:00:00,0.200142,0.009700,0.029884,0.028663,56.733094,9.443696,6.7
2,2025-10-10 01:00:00,0.141956,0.006256,0.027972,0.026023,61.391208,8.153271,6.9
3,2025-10-10 02:00:00,0.139734,0.004469,0.029167,0.023799,61.718539,7.324351,6.9
4,2025-10-10 03:00:00,0.118804,0.003812,0.030145,0.022405,63.611932,6.749256,7.0


In [99]:
n_obs = len(pema_combined)
start_date = pema_combined["datetime_utc"].min()
end_date = pema_combined["datetime_utc"].max()

print(f"Number of observations: {n_obs}")
print(f"Start date: {start_date}")
print(f"End date: {end_date}")

Number of observations: 5197
Start date: 2025-10-09 23:00:00
End date: 2026-05-31 23:00:00


In [100]:
pema_low_rsd = pema_combined[
    pema_combined["datetime_utc"].isin(low_rsd_times)
].copy()

print(f"Rows retained: {len(pema_low_rsd)}")

pema_low_rsd.to_csv("ML ready dpp no2 datasets/pema_no2_train.csv", index=False)
pema_low_rsd.head()

Rows retained: 521


,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,pema_quant_no2
217,2025-11-05 03:00:00,0.186028,0.008567,0.034175,0.024188,56.399595,7.164030,24.8
219,2025-11-05 05:00:00,0.221902,0.007692,0.035977,0.021899,60.930537,6.236489,23.3
263,2025-11-07 01:00:00,0.114096,0.004957,0.030482,0.025177,47.266525,7.211341,21.4
272,2025-11-07 10:00:00,0.170172,0.006503,0.041226,0.019625,64.657456,2.360698,20.5
312,2025-11-09 02:00:00,0.359211,0.008473,0.039159,0.022607,52.321455,11.262695,26.5


### PHA

In [101]:
pha_beacon.head()

,local_timestamp,epoch,datetime,node_file_id,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,node_id
0,2025-10-08 14:00:00,1.759957e+09,2025-10-08 21:00:00,6264048,0.057173,0.000786,0.000611,0.018431,50.194357,19.879838,257
1,2025-10-08 15:00:00,1.759961e+09,2025-10-08 22:00:00,6264336,0.071484,0.005056,-0.002376,0.022234,54.802413,17.822164,257
2,2025-10-08 16:00:00,1.759964e+09,2025-10-08 23:00:00,6264764,0.069348,0.004609,-0.003240,0.022739,54.059721,15.930298,257
3,2025-10-08 17:00:00,1.759968e+09,2025-10-09 00:00:00,6265066,0.080870,0.006531,-0.004611,0.023973,56.990568,14.192671,257
4,2025-10-08 18:00:00,1.759972e+09,2025-10-09 01:00:00,6265689,0.090556,0.008189,-0.006597,0.025731,65.629272,12.550153,257


In [102]:
pha_beacon = pha_beacon[["datetime", "co_wrk_aux", "no2_wrk_aux", "no_wrk_aux", "o3_wrk_aux", "rh", "temp"]].rename(columns={"datetime": "datetime_utc"})
pha_beacon["datetime_utc"] = pd.to_datetime(pha_beacon["datetime_utc"])
pha_beacon.head()

,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp
0,2025-10-08 21:00:00,0.057173,0.000786,0.000611,0.018431,50.194357,19.879838
1,2025-10-08 22:00:00,0.071484,0.005056,-0.002376,0.022234,54.802413,17.822164
2,2025-10-08 23:00:00,0.069348,0.004609,-0.003240,0.022739,54.059721,15.930298
3,2025-10-09 00:00:00,0.080870,0.006531,-0.004611,0.023973,56.990568,14.192671
4,2025-10-09 01:00:00,0.090556,0.008189,-0.006597,0.025731,65.629272,12.550153


In [103]:
pha_quant = no2[["datetime_utc", "pha"]].rename(columns={"pha": "pha_quant_no2"})

pha_combined = pd.merge(pha_beacon, pha_quant, on="datetime_utc", how="inner")

pha_combined = pha_combined.dropna(axis=0)

pha_combined = (pha_combined.sort_values("datetime_utc").reset_index(drop=True))

pha_combined.to_csv("ML ready dpp no2 datasets/pha_no2_test.csv", index=False)
pha_combined.head()

,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,pha_quant_no2
0,2025-10-09 22:00:00,0.110780,0.013482,-0.010464,0.035644,34.219366,12.181335,27.7
1,2025-10-09 23:00:00,0.181230,0.016432,-0.008779,0.033254,46.053264,9.758677,33.4
2,2025-10-10 00:00:00,0.151096,0.015155,-0.010651,0.032106,59.531434,8.312611,28.0
3,2025-10-10 01:00:00,0.144293,0.013782,-0.009632,0.029259,67.073004,7.112694,24.8
4,2025-10-10 02:00:00,0.133378,0.012264,-0.009980,0.026965,72.657811,6.200664,21.5


In [104]:
n_obs = len(pha_combined)
start_date = pha_combined["datetime_utc"].min()
end_date = pha_combined["datetime_utc"].max()

print(f"Number of observations: {n_obs}")
print(f"Start date: {start_date}")
print(f"End date: {end_date}")

Number of observations: 5610
Start date: 2025-10-09 22:00:00
End date: 2026-05-31 23:00:00


In [105]:
pha_low_rsd = pha_combined[
    pha_combined["datetime_utc"].isin(low_rsd_times)
].copy()

print(f"Rows retained: {len(pha_low_rsd)}")

pha_low_rsd.to_csv("ML ready dpp no2 datasets/pha_no2_train.csv", index=False)
pha_low_rsd.head()

Rows retained: 521


,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,pha_quant_no2
629,2025-11-05 03:00:00,0.071157,0.009853,-0.008334,0.030541,44.815477,7.941963,23.9
631,2025-11-05 05:00:00,0.099697,0.010621,-0.007024,0.028548,52.976459,6.312507,24.5
675,2025-11-07 01:00:00,0.063113,0.009445,-0.008410,0.028653,47.157991,6.157044,22.5
684,2025-11-07 10:00:00,0.095790,0.010065,-0.005211,0.021746,69.483370,0.460850,22.1
724,2025-11-09 02:00:00,0.288949,0.013516,0.001050,0.023976,64.115080,9.246057,28.4


## The following analysis includes the MJF quant in the identification of low RSD time periods

In [106]:
low_rsd_dppm = pd.read_csv("low RSD data/no2_low_rsd_dppm.csv")
low_rsd_dppm["datetime_utc"] = pd.to_datetime(low_rsd_dppm["datetime_utc"])

### DPW

In [107]:
#note here we don't have to reconstruct the full test set
#only difference is the training data

low_rsd_times = low_rsd_dppm["datetime_utc"]

dpw_low_rsd = dpw_combined[
    dpw_combined["datetime_utc"].isin(low_rsd_times)
].copy()

print(f"Rows retained: {len(dpw_low_rsd)}")

dpw_low_rsd.to_csv("ML ready dppm no2 datasets/dpw_no2_train.csv", index=False)
dpw_low_rsd.head()

Rows retained: 520


,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,dpw_quant_no2
328,2025-11-07 09:00:00,0.070441,-0.010722,0.022019,0.005506,48.226781,4.766517,21.0
329,2025-11-07 10:00:00,0.083511,-0.010479,0.023948,0.004475,48.717783,4.684071,21.0
362,2025-11-08 19:00:00,0.042153,-0.027061,0.031987,-0.003885,25.255942,27.783985,6.0
630,2025-11-19 23:00:00,0.141408,-0.007792,0.021222,0.009834,36.083613,8.951590,29.9
631,2025-11-20 00:00:00,0.255877,-0.005828,0.028104,0.007629,41.310754,7.956943,34.0


### MJF

In [108]:
mjf_low_rsd = mjf_combined[
    mjf_combined["datetime_utc"].isin(low_rsd_times)
].copy()

print(f"Rows retained: {len(mjf_low_rsd)}")

mjf_low_rsd.to_csv("ML ready dppm no2 datasets/mjf_no2_train.csv", index=False)
mjf_low_rsd.head()

Rows retained: 513


,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,mjf_quant_no2
683,2025-11-07 09:00:00,0.050013,0.020799,0.026609,0.023084,52.886707,2.301982,17.3
684,2025-11-07 10:00:00,0.057022,0.020609,0.026846,0.023342,55.115621,2.068967,16.6
717,2025-11-08 19:00:00,0.031136,0.003587,0.039307,0.022158,22.993522,29.323942,4.5
985,2025-11-19 23:00:00,0.177236,0.025031,0.029177,0.025558,41.009184,4.949780,25.6
986,2025-11-20 00:00:00,0.340136,0.025588,0.040632,0.024671,46.647911,3.355174,24.2


### PEMA

In [109]:
pema_low_rsd = pema_combined[
    pema_combined["datetime_utc"].isin(low_rsd_times)
].copy()

print(f"Rows retained: {len(pema_low_rsd)}")

pema_low_rsd.to_csv("ML ready dppm no2 datasets/pema_no2_train.csv", index=False)
pema_low_rsd.head()

Rows retained: 520


,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,pema_quant_no2
271,2025-11-07 09:00:00,0.162246,0.007016,0.041426,0.020164,67.415214,1.808264,20.1
272,2025-11-07 10:00:00,0.170172,0.006503,0.041226,0.019625,64.657456,2.360698,20.5
305,2025-11-08 19:00:00,0.056435,-0.009650,0.035300,0.016584,27.082833,26.460968,5.3
573,2025-11-19 23:00:00,0.365550,0.012687,0.045673,0.025694,45.095033,5.732753,32.1
574,2025-11-20 00:00:00,0.351666,0.012443,0.052445,0.024110,52.658519,4.090886,29.4


### PHA

In [110]:
pha_low_rsd = pha_combined[
    pha_combined["datetime_utc"].isin(low_rsd_times)
].copy()

print(f"Rows retained: {len(pha_low_rsd)}")

pha_low_rsd.to_csv("ML ready dppm no2 datasets/pha_no2_train.csv", index=False)
pha_low_rsd.head()

Rows retained: 520


,datetime_utc,co_wrk_aux,no2_wrk_aux,no_wrk_aux,o3_wrk_aux,rh,temp,pha_quant_no2
683,2025-11-07 09:00:00,0.125037,0.010594,-0.001019,0.022847,69.758467,0.543361,23.2
684,2025-11-07 10:00:00,0.095790,0.010065,-0.005211,0.021746,69.483370,0.460850,22.1
717,2025-11-08 19:00:00,0.037738,-0.004395,0.006973,0.015989,25.552093,27.626185,5.8
985,2025-11-19 23:00:00,0.294857,0.016536,0.002896,0.027200,48.263436,3.989128,37.0
986,2025-11-20 00:00:00,0.264487,0.015017,0.001261,0.025768,54.164378,2.756168,32.8
